# Intermediate SQL Cont.

## Creating an example database

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sqlite3
import matplotlib.pyplot as plt

## connect with a database / create a new example database
connection_example = sqlite3.connect('new_example.db')
cursor = connection_example.cursor()

In [2]:
## table 1: customers
id_list = [1,2,3,4,5]
income_list = [np.nan, 30, 30, 20, 20]
married_list = ['N', 'Y', 'N', 'Y', 'N']

df = pd.DataFrame(list(zip(id_list, income_list, married_list)),
                  columns =['id', 'income', 'married'])

# delete a table if the table exists in the database
cursor.execute("DROP TABLE IF EXISTS customers;")

# write the DataFrame df to a table called 'customer' in the database 
df.to_sql('customers', connection_example, index=False)

# have a look at the table
table_customers = pd.read_sql("SELECT * FROM customers", connection_example)
print(table_customers.shape) 
table_customers  ## what do you observe? 

(5, 3)


,id,income,married
0,1,NaN,N
1,2,30.0,Y
2,3,30.0,N
3,4,20.0,Y
4,5,20.0,N


In [3]:
# schema details (metadata) of table customers
# e.g., column id (order in table), column name, data type, whether a column should be NOT NULL
# e.g., whether there is any default value, whether a column is part of primary key
table_info_customers = pd.read_sql("PRAGMA table_info(customers);", connection_example)  
table_info_customers

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,income,REAL,0,None,0
2,2,married,TEXT,0,None,0


In [4]:
## table 2: orders
order_id_list = [1,2,3,4,5,6]
customer_id_list = [2,2,1,np.nan,np.nan,1]
amount_list = [10, 30, np.nan, 30, np.nan,30]
date_list = ['2026-01-15','2026-01-30', '2026-01-15', '2026-01-30', '2026-01-15', '2026-01-30']
text_amount_list = ['10.0', '30.0', np.nan, '30.0', np.nan, '30']

df = pd.DataFrame(list(zip(order_id_list, customer_id_list, amount_list, date_list, text_amount_list)),
                  columns =['order_id', 'customer_id', 'amount', 'date', 'text_amount'])

# delete a table if the table exists in the database
cursor.execute("DROP TABLE IF EXISTS orders;")

# write the DataFrame df to a table called 'order' in the database
df.to_sql('orders', connection_example, index=False, dtype={'date': 'DATE'})

# have a look at the table
table_orders = pd.read_sql("SELECT * FROM orders", connection_example)
print(table_orders.shape)
table_orders

(6, 5)


,order_id,customer_id,amount,date,text_amount
0,1,2.0,10.0,2026-01-15,10.0
1,2,2.0,30.0,2026-01-30,30.0
2,3,1.0,NaN,2026-01-15,None
3,4,NaN,30.0,2026-01-30,30.0
4,5,NaN,NaN,2026-01-15,None
5,6,1.0,30.0,2026-01-30,30


In [5]:
# schema details (metadata) of table orders
table_info_orders = pd.read_sql("PRAGMA table_info(orders);", connection_example)
table_info_orders

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,0
1,1,customer_id,REAL,0,None,0
2,2,amount,REAL,0,None,0
3,3,date,DATE,0,None,0
4,4,text_amount,TEXT,0,None,0


In [6]:
## table 3: orders_table
order_id_list = [1,2,3,4,5,6]
customer_id_list = [2,2,1,np.nan,np.nan,1]
amount_list = [10, 30, np.nan, 30, np.nan,30]
date_list = ['2026-01-15','2026-01-30', '2026-01-15', '2026-01-30', '2026-01-15', '2026-01-30']
text_amount_list = ['10.0', '30.0', np.nan, '30.0', np.nan, '30']

df = pd.DataFrame(list(zip(order_id_list, customer_id_list, amount_list, date_list, text_amount_list)),
                  columns =['order_id', 'customer_id', 'amount', 'date', 'text_amount'])

# delete a table if the table exists in the database
cursor.execute("DROP TABLE IF EXISTS orders_table;")

# write the DataFrame df to a table called 'order' in the database
df.to_sql('orders_table', connection_example, index=False, dtype={'date': 'DATE'})

# have a look at the table
table_orders = pd.read_sql("SELECT * FROM orders_table", connection_example)
print(table_orders.shape)
table_orders

(6, 5)


,order_id,customer_id,amount,date,text_amount
0,1,2.0,10.0,2026-01-15,10.0
1,2,2.0,30.0,2026-01-30,30.0
2,3,1.0,NaN,2026-01-15,None
3,4,NaN,30.0,2026-01-30,30.0
4,5,NaN,NaN,2026-01-15,None
5,6,1.0,30.0,2026-01-30,30


In [7]:
# have a look at the database
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_example)
tables

,type,name,tbl_name,rootpage,sql
0,table,customers,customers,2,"CREATE TABLE ""customers"" (\n""id"" INTEGER,\n ""..."
1,table,orders,orders,3,"CREATE TABLE ""orders"" (\n""order_id"" INTEGER,\n..."
2,table,orders_table,orders_table,4,"CREATE TABLE ""orders_table"" (\n""order_id"" INTE..."


In [8]:
# ALTER TABLE: Can be used to add, delete, rename columns, etc. 
cursor.execute("""ALTER TABLE orders
                  RENAME TO orders_new;""")

tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_example)
tables

,type,name,tbl_name,rootpage,sql
0,table,customers,customers,2,"CREATE TABLE ""customers"" (\n""id"" INTEGER,\n ""..."
1,table,orders_new,orders_new,3,"CREATE TABLE ""orders_new"" (\n""order_id"" INTEGE..."
2,table,orders_table,orders_table,4,"CREATE TABLE ""orders_table"" (\n""order_id"" INTE..."


In [9]:
# ALTER TABLE: Can be used to add, delete, rename columns, etc. 
cursor.execute("""ALTER TABLE orders_new
                  ADD new_col INTEGER;""")
table_info_orders = pd.read_sql("PRAGMA table_info(orders_new);", connection_example)
table_info_orders

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,0
1,1,customer_id,REAL,0,None,0
2,2,amount,REAL,0,None,0
3,3,date,DATE,0,None,0
4,4,text_amount,TEXT,0,None,0
5,5,new_col,INTEGER,0,None,0


In [10]:
# delete a table if the table exists in the database
cursor.execute("DROP TABLE IF EXISTS orders_new;")

In [11]:
# DELETE Rows: To delete existing records in a table
cursor.execute("""DELETE FROM orders_table
                  WHERE order_id = 6;
                  """)

tables = pd.read_sql("""SELECT *
                        FROM orders_table;
                        """, connection_example)
tables.head()

,order_id,customer_id,amount,date,text_amount
0,1,2.0,10.0,2026-01-15,10.0
1,2,2.0,30.0,2026-01-30,30.0
2,3,1.0,NaN,2026-01-15,None
3,4,NaN,30.0,2026-01-30,30.0
4,5,NaN,NaN,2026-01-15,None


In [12]:
# UPDATE Table: To modify the existing records in a table.
cursor.execute("""UPDATE orders_table
                  SET customer_id = 1
                  WHERE order_id = 1;
                  """)

tables = pd.read_sql("""SELECT *
                        FROM orders_table;
                        """, connection_example)
tables.head()

,order_id,customer_id,amount,date,text_amount
0,1,1.0,10.0,2026-01-15,10.0
1,2,2.0,30.0,2026-01-30,30.0
2,3,1.0,NaN,2026-01-15,None
3,4,NaN,30.0,2026-01-30,30.0
4,5,NaN,NaN,2026-01-15,None


In [13]:
## commit the current transaction / change
## permanently save all data modifications (e.g., INSERT, UPDATE, DELETE) during the current session to the database
connection_example.commit()  

In [14]:
connection_example.close() 

In [15]:
## connect with the database
connection_example = sqlite3.connect('new_example.db')
cursor = connection_example.cursor()

## metadata about the tables
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_example)
tables

,type,name,tbl_name,rootpage,sql
0,table,customers,customers,2,"CREATE TABLE ""customers"" (\n""id"" INTEGER,\n ""..."
1,table,orders_table,orders_table,4,"CREATE TABLE ""orders_table"" (\n""order_id"" INTE..."


In [16]:
## have a look at customers table
customers = pd.read_sql("""SELECT *
                        FROM customers;""", connection_example)
customers

,id,income,married
0,1,NaN,N
1,2,30.0,Y
2,3,30.0,N
3,4,20.0,Y
4,5,20.0,N


In [17]:
table_info_customers = pd.read_sql("PRAGMA table_info(customers);", connection_example)
table_info_customers

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,0
1,1,income,REAL,0,None,0
2,2,married,TEXT,0,None,0


In [18]:
## have a look at orders_table 
orders = pd.read_sql("""SELECT *
                        FROM orders_table;""", connection_example)
orders

,order_id,customer_id,amount,date,text_amount
0,1,1.0,10.0,2026-01-15,10.0
1,2,2.0,30.0,2026-01-30,30.0
2,3,1.0,NaN,2026-01-15,None
3,4,NaN,30.0,2026-01-30,30.0
4,5,NaN,NaN,2026-01-15,None


In [19]:
table_info_orders = pd.read_sql("PRAGMA table_info(orders_table);", connection_example)
table_info_orders

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,0
1,1,customer_id,REAL,0,None,0
2,2,amount,REAL,0,None,0
3,3,date,DATE,0,None,0
4,4,text_amount,TEXT,0,None,0


In [20]:
## have a look at orders_table
orders = pd.read_sql("""SELECT amount
                        FROM orders_table;""", connection_example)
orders

,amount
0,10.0
1,30.0
2,NaN
3,30.0
4,NaN


## Set Operations

#### INTERSECT

In [21]:
# INTERSECT
result1 = pd.read_sql("""SELECT id
                         FROM customers
                         INTERSECT
                         SELECT customer_id 
                         FROM orders_table;
                         """, connection_example)
result1

,id
0,1
1,2


In [22]:
# INTERSECT
result1_1 = pd.read_sql("""SELECT id      -- does it make sense?        
                         FROM customers
                         INTERSECT
                         SELECT date 
                         FROM orders_table;
                         """, connection_example)
result1_1

,id


In [23]:
# INTERSECT
result1_2 = pd.read_sql("""SELECT id, income  
                         FROM customers
                         INTERSECT
                         SELECT customer_id, amount
                         FROM orders_table;
                         """, connection_example)
result1_2

,id,income
0,1,NaN
1,2,30.0


In [24]:
# INTERSECT
result1_2 = pd.read_sql("""SELECT income   
                         FROM customers
                         INTERSECT
                         SELECT text_amount
                         FROM orders_table;
                         """, connection_example)
result1_2

,income
0,None


In [25]:
# INTERSECT
result1_2 = pd.read_sql("""SELECT id, income   
                         FROM customers
                         INTERSECT
                         SELECT customer_id, text_amount
                         FROM orders_table;
                         """, connection_example)
result1_2

,id,income
0,1,None


In [26]:
# INTERSECT
result1_3 = pd.read_sql("""SELECT id
                         FROM customers
                         INTERSECT
                         SELECT customer_id, amount
                         FROM orders_table;
                         """, connection_example)
result1_3

DatabaseError: Execution failed on sql 'SELECT id
                         FROM customers
                         INTERSECT
                         SELECT customer_id, amount
                         FROM orders_table;
                         ': SELECTs to the left and right of INTERSECT do not have the same number of result columns

#### EXCEPT

In [27]:
## EXCEPT
result2 = pd.read_sql("""SELECT id
                         FROM customers
                         EXCEPT
                         SELECT customer_id
                         FROM orders_table;
                         """, connection_example)
result2

,id
0,3
1,4
2,5


In [28]:
## EXCEPT
result2_1 = pd.read_sql("""SELECT income
                         FROM customers
                         EXCEPT
                         SELECT date
                         FROM orders_table;
                         """, connection_example)
result2_1

,income
0,NaN
1,20.0
2,30.0


In [29]:
## EXCEPT
result2_2 = pd.read_sql("""SELECT id, income
                         FROM customers
                         EXCEPT
                         SELECT customer_id, amount
                         FROM orders_table;
                         """, connection_example)
result2_2

,id,income
0,3,30.0
1,4,20.0
2,5,20.0


#### UNION

In [30]:
## UNION
result3 = pd.read_sql("""SELECT id
                         FROM customers
                         UNION
                         SELECT customer_id
                         FROM orders_table;
                         """, connection_example)
result3

,id
0,NaN
1,1.0
2,2.0
3,3.0
4,4.0
5,5.0


In [32]:
## UNION
result3_1 = pd.read_sql("""SELECT id
                         FROM customers
                         UNION
                         SELECT date
                         FROM orders_table;
                         """, connection_example)
result3_1

,id
0,1
1,2
2,3
3,4
4,5
5,2026-01-15
6,2026-01-30


In [33]:
## UNION
result3_2 = pd.read_sql("""SELECT id, income
                         FROM customers
                         UNION
                         SELECT customer_id, amount
                         FROM orders_table;
                         """, connection_example)
result3_2

,id,income
0,NaN,NaN
1,NaN,30.0
2,1.0,NaN
3,1.0,10.0
4,2.0,30.0
5,3.0,30.0
6,4.0,20.0
7,5.0,20.0


#### UNION ALL

In [34]:
## UNION ALL
result4 = pd.read_sql("""SELECT id
                         FROM customers
                         UNION ALL
                         SELECT customer_id
                         FROM orders_table;
                         """, connection_example)
result4

,id
0,1.0
1,2.0
2,3.0
3,4.0
4,5.0
5,1.0
6,2.0
7,1.0
8,NaN
9,NaN


In [35]:
## UNION ALL
result4_1 = pd.read_sql("""SELECT id
                         FROM customers
                         UNION ALL
                         SELECT date
                         FROM orders_table;
                         """, connection_example)
result4_1

,id
0,1
1,2
2,3
3,4
4,5
5,2026-01-15
6,2026-01-30
7,2026-01-15
8,2026-01-30
9,2026-01-15


In [36]:
## UNION ALL
result4_2 = pd.read_sql("""SELECT id, income
                         FROM customers
                         UNION ALL
                         SELECT customer_id, amount
                         FROM orders_table;
                         """, connection_example)
result4_2

,id,income
0,1.0,NaN
1,2.0,30.0
2,3.0,30.0
3,4.0,20.0
4,5.0,20.0
5,1.0,10.0
6,2.0,30.0
7,1.0,NaN
8,NaN,30.0
9,NaN,NaN


## Joins

In [37]:
# Cartesian Product or Cross Join
result5 = pd.read_sql("""SELECT *
                         FROM customers, orders_table;
                         """, connection_example)
print(result5.shape)
result5

(25, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,1,NaN,N,1,1.0,10.0,2026-01-15,10.0
1,1,NaN,N,2,2.0,30.0,2026-01-30,30.0
2,1,NaN,N,3,1.0,NaN,2026-01-15,None
3,1,NaN,N,4,NaN,30.0,2026-01-30,30.0
4,1,NaN,N,5,NaN,NaN,2026-01-15,None
5,2,30.0,Y,1,1.0,10.0,2026-01-15,10.0
6,2,30.0,Y,2,2.0,30.0,2026-01-30,30.0
7,2,30.0,Y,3,1.0,NaN,2026-01-15,None
8,2,30.0,Y,4,NaN,30.0,2026-01-30,30.0
9,2,30.0,Y,5,NaN,NaN,2026-01-15,None


In [38]:
# INNER JOIN
result6 = pd.read_sql("""SELECT *
                         FROM customers c
                         JOIN orders_table o
                         ON c.id = o.customer_id;   
                         """, connection_example)
print(result6.shape)
result6

(3, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,1,NaN,N,1,1.0,10.0,2026-01-15,10.0
1,1,NaN,N,3,1.0,NaN,2026-01-15,None
2,2,30.0,Y,2,2.0,30.0,2026-01-30,30.0


In [39]:
# INNER JOIN
result6_1 = pd.read_sql("""SELECT *
                         FROM customers c
                         JOIN orders_table o
                         ON c.income = o.amount;  -- o.text_amount
                         """, connection_example)
print(result6_1.shape)
result6_1

(4, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,2,30.0,Y,2,2.0,30.0,2026-01-30,30.0
1,2,30.0,Y,4,NaN,30.0,2026-01-30,30.0
2,3,30.0,N,2,2.0,30.0,2026-01-30,30.0
3,3,30.0,N,4,NaN,30.0,2026-01-30,30.0


In [40]:
# INNER JOIN
result7 = pd.read_sql("""SELECT *
                         FROM customers c
                         JOIN orders_table o
                         ON c.id = o.customer_id AND c.income = o.amount;
                         """, connection_example)
print(result7.shape)
result7

(1, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,2,30.0,Y,2,2.0,30.0,2026-01-30,30.0


In [41]:
# LEFT JOIN
result8 = pd.read_sql("""SELECT *
                         FROM customers c
                         LEFT JOIN orders_table o
                         ON c.id = o.customer_id AND c.income = o.amount;
                         """, connection_example)
print(result8.shape)
result8

(5, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,1,NaN,N,NaN,NaN,NaN,None,None
1,2,30.0,Y,2.0,2.0,30.0,2026-01-30,30.0
2,3,30.0,N,NaN,NaN,NaN,None,None
3,4,20.0,Y,NaN,NaN,NaN,None,None
4,5,20.0,N,NaN,NaN,NaN,None,None


## Set Operations vs Joins

#### INTERSECT vs JOIN

In [42]:
result1_1 = pd.read_sql("""SELECT income             
                         FROM customers
                         INTERSECT
                         SELECT amount 
                         FROM orders_table;
                         """, connection_example)
result1_1

,income
0,NaN
1,30.0


In [43]:
# INTERSECT vs JOIN
result1_2 = pd.read_sql("""SELECT DISTINCT c.income
                         FROM customers c
                         JOIN orders_table o
                         ON c.income = o.amount;
                         """, connection_example)
print(result1_2.shape)
result1_2

(1, 1)


,income
0,30.0


In [44]:
# INTERSECT vs JOIN
result1_3 = pd.read_sql("""SELECT c.income
                         FROM customers c
                         JOIN orders_table o
                         ON c.income = o.amount;
                         """, connection_example)
print(result1_3.shape)
result1_3

(4, 1)


,income
0,30.0
1,30.0
2,30.0
3,30.0


In [45]:
# INTERSECT vs JOIN
result1_4 = pd.read_sql("""SELECT *
                         FROM customers c
                         JOIN orders_table o
                         ON c.income = o.amount;
                         """, connection_example)
print(result1_4.shape)
result1_4

(4, 8)


,id,income,married,order_id,customer_id,amount,date,text_amount
0,2,30.0,Y,2,2.0,30.0,2026-01-30,30.0
1,2,30.0,Y,4,NaN,30.0,2026-01-30,30.0
2,3,30.0,N,2,2.0,30.0,2026-01-30,30.0
3,3,30.0,N,4,NaN,30.0,2026-01-30,30.0


#### EXCEPT vs NOT IN

In [46]:
## EXCEPT
result2_1 = pd.read_sql("""SELECT income
                         FROM customers
                         EXCEPT
                         SELECT amount
                         FROM orders_table;
                         """, connection_example)
result2_1

,income
0,20.0


In [47]:
## EXCEPT vs NOT IN
result2_2 = pd.read_sql("""SELECT income
                         FROM customers
                         WHERE income NOT IN 
                               (SELECT amount
                                FROM orders_table);
                         """, connection_example)
result2_2

,income


In [48]:
## EXCEPT vs NOT IN
result2_2 = pd.read_sql("""SELECT amount
                           FROM orders_table;
                         """, connection_example)
result2_2

,amount
0,10.0
1,30.0
2,NaN
3,30.0
4,NaN


In [49]:
## EXCEPT vs NOT IN
result2_3 = pd.read_sql("""SELECT *
                         FROM customers
                         WHERE income NOT IN 
                               (SELECT amount
                                FROM orders_table
                                WHERE amount IS NOT NULL);
                         """, connection_example)
result2_3

,id,income,married
0,4,20.0,Y
1,5,20.0,N


In [50]:
## EXCEPT vs NOT IN
result2_3 = pd.read_sql("""SELECT amount
                                FROM orders_table
                                WHERE amount IS NOT NULL;
                         """, connection_example)
result2_3

,amount
0,10.0
1,30.0
2,30.0


In [51]:
## EXCEPT vs NOT IN
result2_4 = pd.read_sql("""SELECT DISTINCT income
                         FROM customers
                         WHERE income NOT IN 
                               (SELECT amount
                                FROM orders_table
                                WHERE amount IS NOT NULL);
                         """, connection_example)
result2_4

,income
0,20.0


## If finishing working with the database

In [52]:
connection_example.close() 

In [53]:
# delete the database file
import os
os.remove('new_example.db')